In [402]:
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid
import pandas as pd
import numpy as np
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
from tespy.tools import UserDefinedEquation

In [403]:

class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_ext_id,HC_inj_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_ext_id=HC_ext_id
        self.HC_inj_id=HC_inj_id
        self.net=net
        
    def _build_tespy_Cycle(self,mode):
        self.mode=mode
        self.nw = Network()
        self.nw.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")

        self.compressor = Compressor("compresor")
        self.condenser = Condenser("condensador")
        self.valve = Valve("valvula_expansion")
        self.evaporator = HeatExchanger("evaporador")
        self.cc=CycleCloser('CycleCloser')
        self.cc_2=CycleCloser('CycleCloser_Consumer_side')
        self.Consumer=SimpleHeatExchanger("Consumer")
        self.source=Source("source")
        self.sink=Sink("Sink")
        #Refirgerant Side
        self.c0=Connection(self.valve, 'out1', self.cc, 'in1', label='0')
        self.c1 = Connection(self.cc, 'out1', self.evaporator, 'in2', label='1')
        self.c2 = Connection(self.evaporator, 'out2', self.compressor, 'in1', label='2')
        self.c3 = Connection(self.compressor, 'out1', self.condenser, 'in1', label='3')
        self.c4 = Connection(self.condenser, 'out1', self.valve, 'in1', label='4')
        if self.mode=="COOLING_NET":
            self.c5=Connection(self.condenser, 'out2',self.Consumer, 'in1', label='5')
            self.c6=Connection(self.Consumer, 'out1',self.cc_2 , 'in1', label='6')
            self.c7=Connection(self.cc_2, 'out1',self.condenser , 'in2', label='7')

            #Connection district heating 
            self.c8=Connection(self.source, 'out1',self.evaporator , 'in1', label='8')
            self.c9=Connection(self.evaporator, 'out1',self.sink , 'in1', label='9')
        else: 
            self.c5 = Connection(self.evaporator, 'out1', self.Consumer, 'in1', label='5')
            self.c6 = Connection(self.Consumer, 'out1', self.cc_2, 'in1', label='6')
            self.c7 = Connection(self.cc_2, 'out1', self.evaporator, 'in1', label='7')  # Corrección de puertos

            # 5. Circuito de la Red (Ahora actúa como SUMIDERO de calor -> va al Condensador)
            self.c8 = Connection(self.source, 'out1', self.condenser, 'in2', label='8')
            self.c9 = Connection(self.condenser, 'out2', self.sink, 'in1', label='9')
        self.nw.add_conns(self.c0,  self.c1,  self.c2,  self.c3,  self.c4 , self.c5,  self.c6, self.c7, self.c8,self.c9)
        def my_ude(ude):
            return ude.conns[0].calc_T_dew() +5-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        ude = UserDefinedEquation(
        'my ude', my_ude, my_ude_dependents, conns=[self.c1, self.c2])
        self.nw.add_ude(ude)
        

    def solve_cycle(self,net,Q_know,eta_s,T_nework,T_cons):
        #Dt= Heating network 
        if self.mode=="STATIC":
            print(f"{self.name} is STATIC")
        else:
            T_cons_in = T_cons[0]
            T_cons_out = T_cons[1]
            T_DH_in = T_nework[0]
            T_DH_out = T_nework[1]
            self.evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.compressor.set_attr(eta_s=eta_s)
            self.c2.set_attr(fluid={self.refrigerant: 1})
            # 6. Parámetros del Consumidor 
            self.c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.c7.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.c8.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            self.c9.set_attr(T=T_DH_out)
            self.Consumer.set_attr(Q=Q_know)
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw._set_p_range([p_triple, p_high])
            self.nw._set_h_range([h_min,h_max])
            self.nw.solve('design')
    
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =True
            self.net.heat_consumer.at[self.HC_ext_id, "qext_w"] =abs(self.evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_ext_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =True
            self.net.heat_consumer.at[self.HC_inj_id, "qext_w"] =self.condenser.Q.val
            self.net.heat_consumer.at[self.HC_inj_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False
        else: 
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False


In [404]:
class ThermoclineTwoLayer():
    def __init__(self, net, name, circ_pump_charge_id,circ_pump_decharge_id, flow_control_charge_id, flow_control_decharge_id, volume_m3, t_hot_init, t_cold_init, v_hot_fraction_init, UA_loss, UA_interface=0.0):
        self.net = net
        self.fluid = get_fluid(net)
        self.name = name
        self.circ_pump_charge_id= circ_pump_charge_id
        self.circ_pump_decharge_id= circ_pump_decharge_id
        self.flow_control_charge_id= flow_control_charge_id
        self.flow_control_decharge_id= flow_control_decharge_id
        self.V_tot = volume_m3
        self.T_hot = t_hot_init
        self.T_cold = t_cold_init
        self.v_hot_fraction = v_hot_fraction_init
        self.UA_loss = UA_loss  # W/K, total heat loss with the ambient
        self.UA_interface = UA_interface  # W/K, exchange between hot and cold layers (mixing/conduction)
        self.V_hot = volume_m3 * self.v_hot_fraction
        self.V_cold = volume_m3 - self.V_hot
    
    def Interaction_Simulation(self,mass_flow):
        'This function will establish the values of sink and source temperatures and mass flows based on the information provided.'
        'Here the convention is if the mass flow is positive, it means that the flow is entering the storage tank and the , and if it is negative, it means that the flow is from the sink to the source.'
        #Condition to be charged or static
        if mass_flow >= 0:
            #Supply junction:
            self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "mdot_flow_kg_per_s"] =0
            self.net.flow_control.at[self.flow_control_decharge_id, "controlled_mdot_kg_per_s"] =0
            self.net.circ_pump_mass.at[self.circ_pump_charge_id, "mdot_flow_kg_per_s"] = mass_flow
            self.net.circ_pump_mass.at[self.circ_pump_charge_id, "t_flow_k"] =self.T_cold
            self.net.flow_control.at[self.flow_control_charge_id, "controlled_mdot_kg_per_s"] =mass_flow
            
        # Discharging the storage tank (mass flow is negative)
        else:
            self.net.circ_pump_mass.at[self.circ_pump_charge_id, "mdot_flow_kg_per_s"] = 0
            self.net.flow_control.at[self.flow_control_charge_id, "controlled_mdot_kg_per_s"] =0
            self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "mdot_flow_kg_per_s"] = abs(mass_flow)
            self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "t_flow_k"] =self.T_hot
            self.net.flow_control.at[self.flow_control_decharge_id,"controlled_mdot_kg_per_s"]= abs(mass_flow)
    def Estimate_loss_ambient(self,T_amb):
        self.UA_hot_layer=self.UA_loss*(self.V_hot / self.V_tot)
        self.UA_cold_layer=self.UA_loss*(self.V_cold / self.V_tot)
        Q_loss_hot = self.UA_hot_layer * (self.T_hot - T_amb)
        Q_loss_cold = self.UA_cold_layer * (self.T_cold - T_amb)
        return Q_loss_hot,Q_loss_cold
            
    def V_dis(self, dt_s,T_amb,mass_flow):
        # Calculate the volume displaced to the mass flows interaction and time step
        self.UA_hot_layer=self.UA_loss*(self.V_hot / self.V_tot)
        self.UA_cold_layer=self.UA_loss*(self.V_cold / self.V_tot)
        #Charging the storage tank (mass flow is positive)
        V_MIN = 1e-4
        if mass_flow >=0:
            T_net = self.net.res_circ_pump_mass.at[self.circ_pump_charge_id, "t_from_k"]
            Dv = mass_flow * dt_s / self.fluid.get_density(self.T_hot)
            v_in = min(Dv, self.V_cold-V_MIN)
            v_in = max(v_in, 0.0)

            #Restriction according to the cold volume present at the moment
            if v_in < Dv - 1e-6:
                print(f"  ⚠ [{self.name}] Tanque saturado de calor: "
                  f"{Dv - v_in:.4f} m3 no se pudieron cargar "
                  f"(quedan {V_MIN} m3 de margen frío)") 
            self.T_hot=self.T_hot+dt_s*(((mass_flow*(T_net-self.T_hot))/(self.fluid.get_density(self.T_hot)*self.V_hot))-((self.UA_hot_layer*(self.T_hot-T_amb))/(self.fluid.get_density(self.T_hot)*self.V_hot*self.fluid.get_heat_capacity(self.T_hot))))
            self.T_cold=self.T_cold+dt_s*(-self.UA_cold_layer*(self.T_cold-T_amb)/(self.fluid.get_density(self.T_cold)*self.V_cold*self.fluid.get_heat_capacity(self.T_cold)))
            self.V_hot=self.V_hot+v_in
            self.V_cold=self.V_cold-v_in
            
        # Discharging the storage tank (mass flow is negative)
        else:
            T_net = self.net.res_circ_pump_mass.at[self.circ_pump_decharge_id, "t_from_k"]
            Dv = abs(mass_flow) * dt_s / self.fluid.get_density(self.T_cold)
            v_in= min(Dv, self.V_hot-V_MIN)
            v_in = max(v_in, 0.0)
            #Restriction according to the hot volume present at the moment
            if v_in < Dv - 1e-6:
                print(f"  ⚠ [{self.name}] Tanque saturado de frío: "
                  f"{Dv - v_in:.4f} m3 no se pudieron descargar "
                  f"(quedan {V_MIN} m3 de margen caliente)")
            self.T_cold=self.T_cold+dt_s*((abs(mass_flow)/(self.fluid.get_density(self.T_cold)*self.V_cold))-(self.UA_cold_layer/(self.fluid.get_density(self.T_cold)*self.V_cold*self.fluid.get_heat_capacity(self.T_cold))))  
            self.T_hot=self.T_hot+dt_s*(-(self.UA_hot_layer*(self.T_hot-T_amb)/(self.fluid.get_density(self.T_hot)*self.V_hot*self.fluid.get_heat_capacity(self.T_hot)))) 
            self.V_cold=self.V_cold+v_in
            self.V_hot=self.V_hot-v_in
        self.v_hot_fraction=self.V_hot/self.V_tot

In [405]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_ida_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_Ida")
j_HP_ida_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_Ida")
j_HP_ida_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_Ida")
j_storage=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Storage")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_ret")
j_HP_ret_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_ret")

In [406]:
#Plant-storage
#Planta Nodo
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15), text_k=273.2, name="Tubo_Ida_Plant_Storage_ida",k_mm=0.1*1000
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_RetornoPlant_Storage_ret",k_mm=0.1*1000)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_Storage_ida_nodo_1",k_mm=0.1*1000
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_Storage_ret_nodo_1",k_mm=0.1*1000
)

#Node Consumers 
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_ida_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_1_ida",k_mm=0.1*1000
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_1_ret",k_mm=0.1*1000
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_ida_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_2_ret",k_mm=0.1*1000
)
#Node_Cons3
pipe_ida_5= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_HP_ida_3,
    length_km=0.0441, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_3_ida",k_mm=0.1*1000
)
pipe_retorno_5= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_3, to_junction=j_nodo_2_ret,
    length_km=0.0441, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_3_ret",k_mm=0.1*1000
)

In [407]:
#Heat Pumps
#Heat pump1: extraction
HP_1_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ida_1,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat mump 1: Injection
HP_1_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_1,
    to_junction=j_HP_ida_1,
    qext_w=-150000,
    deltat_k=50,
    name="HP_1_INJ"
)
#Heat pump2: extraction
HP_2_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ida_2 ,
    to_junction=j_HP_ret_2,
    qext_w=150000,
    deltat_k=50,
    name="HP_2_EXT"
)
#Heat mump 2: Injection
HP_2_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_2,
    to_junction=j_HP_ida_2,
    qext_w=150000,
    deltat_k=-50,
    name="HP_2_INJ"
)
#Heat pump3: extraction
HP_3_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ida_3 ,
    to_junction=j_HP_ret_3,
    qext_w=150000,
    deltat_k=50,
    name="HP_3_EXT"
)
#Heat mump 2: Injection
HP_3_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_3,
    to_junction=j_HP_ida_3,
    qext_w=150000,
    deltat_k=-50,
    name="HP_3_INJ"
)

In [408]:
#Main plant 
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=340 ,name='Grid')


In [ ]:
# Storage in the first node 
#decharging
Circ_pump_decharge=pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_nodo_1_ret, flow_junction=j_storage,
    mdot_flow_kg_per_s=0.1, t_flow_k=300, p_flow_bar=5  # valor arbitrario, ver nota abajo
)

Flow_control_decharge=pp.create_flow_control(
    net, from_junction=j_storage, to_junction=j_nodo_1,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)
flow_control_bypass_charge=pp.create_flow_control()
#Charging 
Circ_pump_charge=pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_nodo_1, flow_junction=j_storage,
    mdot_flow_kg_per_s=0.1, t_flow_k=300, p_flow_bar=5  # valor arbitrario, ver nota abajo
)

Flow_control_charge=pp.create_flow_control(
    net, from_junction=j_storage, to_junction=j_nodo_1_ret,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)

In [410]:
Storage=ThermoclineTwoLayer(net, "Storage_1", Circ_pump_charge,Circ_pump_decharge, Flow_control_charge, Flow_control_decharge,volume_m3=10, t_hot_init=330, t_cold_init=313, v_hot_fraction_init=2/10,UA_loss=15, UA_interface=0.0)
HP_1=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_ext_id=HP_1_Ext,HC_inj_id=HP_1_inj)
HP_2=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_ext_id=HP_2_Ext,HC_inj_id=HP_2_inj)
HP_3=Bidirectional_W_to_WHeatPump(name="heat_pump_3",refrigerant="R134a",net=net,HC_ext_id=HP_3_Ext,HC_inj_id=HP_3_inj)

In [411]:
Data=pd.read_excel("C:\\Users\\sserranose\\OneDrive - INSA Lyon\\Bureau\\Code\\Panda_pipes\\Profile_mix_case.xlsx")

In [412]:
Data

,H,S,Q HP_1_Consumer,DT,Mode,Q HP_2_Consumer,DT.1,Mode .1,Q HP_3_Consumer,DT.2,Mode .2,Debit Stockage
0,0,0,-15000,5,COOLING_NET,0,0,STATIC,1500,-10,HEATING_NET,0.00
1,1,3600,-15000,5,COOLING_NET,0,0,STATIC,1500,-10,HEATING_NET,-0.88
2,2,7200,-15000,5,COOLING_NET,0,0,STATIC,1500,-10,HEATING_NET,0.15
3,3,10800,-15000,5,COOLING_NET,0,0,STATIC,1500,-10,HEATING_NET,0.18
4,4,14400,0,0,STATIC,0,0,STATIC,0,0,STATIC,-0.88
5,5,18000,0,0,STATIC,0,0,STATIC,0,0,STATIC,-0.75
6,6,21600,1500,-10,HEATING_NET,-15000,5,COOLING_NET,-15000,5,COOLING_NET,-0.75
7,7,25200,1500,-10,HEATING_NET,-15000,5,COOLING_NET,-15000,5,COOLING_NET,0.40
8,8,28800,1500,-10,HEATING_NET,-15000,5,COOLING_NET,-15000,5,COOLING_NET,0.50
9,9,32400,1500,-10,HEATING_NET,-15000,5,COOLING_NET,-15000,5,COOLING_NET,-0.75


In [413]:
Q_HP_consumer_1=Data.iloc[:,2].to_numpy()
DT_HP_1=Data.iloc[:,3].to_numpy()
Mode_Hp_1=Data.iloc[:,4].to_numpy()
Q_HP_consumer_2=Data.iloc[:,5].to_numpy()
DT_HP_2=Data.iloc[:,6].to_numpy()
Mode_Hp_2=Data.iloc[:,7].to_numpy()
Q_HP_consumer_3=Data.iloc[:,8].to_numpy()
DT_HP_3=Data.iloc[:,9].to_numpy()
Mode_Hp_3=Data.iloc[:,10].to_numpy()

In [414]:
T_cons_cooling_net=[50,60]
T_cons_heating_net=[35,25]


In [415]:
Débit_Storage=Data.iloc[:,11].to_numpy()*-1

In [416]:
Débit_Storage

array([-0.  ,  0.88, -0.15, -0.18,  0.88,  0.75,  0.75, -0.4 , -0.5 ,
        0.75, -0.1 ])

In [417]:
for i in range(len(Débit_Storage)):
    print("interation"+ str(i))
    if Mode_Hp_1[i]!= "STATIC": 
        if Mode_Hp_1[i]=="COOLING_NET":
            HP_1.T_network_inital_guess=[35,25]
            HP_1.T_cons=T_cons_cooling_net
        elif Mode_Hp_1[i]=="HEATING_NET":
            HP_1.T_network_inital_guess=[50,60]
            HP_1.T_cons=T_cons_heating_net      
    else:
        HP_1.T_network_inital_guess=[0,0]
        HP_1.T_cons=[0,0]
    if Mode_Hp_2[i]!= "STATIC": 
        if Mode_Hp_2[i]=="COOLING_NET":
            HP_2.T_network_inital_guess=[35,25]
            HP_2.T_cons=T_cons_cooling_net
        elif Mode_Hp_2[i]=="HEATING_NET":
            HP_2.T_network_inital_guess=[50,60]
            HP_2.T_cons=T_cons_heating_net
    else:
        HP_2.T_network_inital_guess=[0,0]
        HP_2.T_cons=[0,0]
    if Mode_Hp_3[i]!= "STATIC": 
        if Mode_Hp_3[i]=="COOLING_NET":
            HP_3.T_network_inital_guess=[35,25]
            HP_3.T_cons=T_cons_cooling_net
        elif Mode_Hp_3[i]=="HEATING_NET":
            HP_3.T_network_inital_guess=[50,60]
            HP_3.T_cons=T_cons_heating_net
    else:
        HP_3.T_network_inital_guess=[0,0]
        HP_3.T_cons=[0,0]
    HP_1._build_tespy_Cycle(mode=Mode_Hp_1[i])
    HP_2._build_tespy_Cycle(mode=Mode_Hp_2[i])   
    HP_3._build_tespy_Cycle(mode=Mode_Hp_3[i])   
    #initial Guess
    pp.pipeflow(net,mode="bidirectional")
    tolerance=1E-6
    error_1=10
    error_2=10
    error_3=10
    T_network_HP_1=HP_1.T_network_inital_guess
    T_network_HP_2=HP_2.T_network_inital_guess
    T_network_HP_3=HP_3.T_network_inital_guess
    T_network_HP_1_loop=HP_1.T_network_inital_guess
    T_network_HP_2_loop=HP_2.T_network_inital_guess
    T_network_HP_3_loop=HP_3.T_network_inital_guess
    while error_1>tolerance or error_2>tolerance or error_3>tolerance:
        HP_1.solve_cycle(net=net,Q_know=Q_HP_consumer_1[i],eta_s=0.95,T_nework=T_network_HP_1,T_cons=HP_1.T_cons)     
        HP_1.interaction_simulation(dT_water=DT_HP_1[i])
        HP_2.solve_cycle(net=net,Q_know=Q_HP_consumer_2[i],eta_s=0.95,T_nework=T_network_HP_2,T_cons=HP_2.T_cons)     
        HP_2.interaction_simulation(dT_water=DT_HP_2[i])
        HP_3.solve_cycle(net=net,Q_know=Q_HP_consumer_3[i],eta_s=0.95,T_nework=T_network_HP_3,T_cons=HP_3.T_cons)     
        HP_3.interaction_simulation(dT_water=DT_HP_3[i])
        Storage.Interaction_Simulation(Débit_Storage[i])
        pp.pipeflow(net,mode="bidirectional")
        T_network_HP_1=[]
        T_network_HP_2=[]
        T_network_HP_3=[]
        if HP_1.mode== "STATIC":
            T_network_HP_1=[0,0]
        elif HP_1.mode == "COOLING_NET":
            T_network_HP_1.append(float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_from_k"]-273.15))
            T_network_HP_1.append(float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_to_k"]-273.15))
        else:  # HEATING_NET
            T_network_HP_1.append(float(net.res_heat_consumer.at[HP_1.HC_inj_id,"t_from_k"]-273.15))
            T_network_HP_1.append(float(net.res_heat_consumer.at[HP_1.HC_inj_id,"t_to_k"]-273.15))
        if HP_2.mode== "STATIC":
            T_network_HP_2=[0,0]
        elif HP_2.mode == "COOLING_NET":
            T_network_HP_2.append(float(net.res_heat_consumer.at[HP_2.HC_ext_id,"t_from_k"]-273.15))
            T_network_HP_2.append(float(net.res_heat_consumer.at[HP_2.HC_ext_id,"t_to_k"]-273.15))
        else:  # HEATING_NET
            T_network_HP_2.append(float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_from_k"]-273.15))
            T_network_HP_2.append(float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_to_k"]-273.15))
        if HP_3.mode== "STATIC":
            T_network_HP_3=[0,0]
        elif HP_3.mode == "COOLING_NET":
            T_network_HP_3.append(float(net.res_heat_consumer.at[HP_3.HC_ext_id,"t_from_k"]-273.15))
            T_network_HP_3.append(float(net.res_heat_consumer.at[HP_3.HC_ext_id,"t_to_k"]-273.15))
        else:  # HEATING_NET
            T_network_HP_3.append(float(net.res_heat_consumer.at[HP_3.HC_inj_id,"t_from_k"]-273.15))
            T_network_HP_3.append(float(net.res_heat_consumer.at[HP_3.HC_inj_id,"t_to_k"]-273.15))
        error_1=abs(T_network_HP_1_loop[0]-T_network_HP_1[0])
        error_2=abs(T_network_HP_2_loop[0]-T_network_HP_2[0])
        error_3=abs(T_network_HP_3_loop[0]-T_network_HP_3[0])
        T_network_HP_1_loop=T_network_HP_1
        T_network_HP_2_loop=T_network_HP_2
        T_network_HP_3_loop=T_network_HP_3
    if Débit_Storage[i]==0: 
        pass
    else:
        Storage.V_dis(dt_s=3600, mass_flow=Débit_Storage[i],T_amb=273.2)
    print(Storage.V_hot)
    print(Storage.T_hot)

interation0

 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 2.05e+05   | 7 %        | 5.03e+01   | 1.61e+06   | 1.55e+06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 9.41e+06   | 0 %        | 1.80e+02   | 3.16e+05   | 1.12e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 5.36e+06   | 0 %        | 1.28e+02   | 1.45e+05   | 3.50e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 5.91e+05   | 2 %        | 4.62e+00   | 3.85e+04   | 2.74e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 5.37e+05   | 2 %        | 3.30e+01   | 1.84e+05   | 2.12e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 1.64e+06   | 0 %        | 3.32e+01   | 3.99e+04   | 5.79e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 7     | 6.57e+03   | 24 %       | 2.04e-01   | 1.15e+03   | 3.42e+01   | 0.00e+00   | 0.00